In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

## Chat completion model

In [2]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-3.5-turbo-0125")

## Prompts and Prompt templates

In [3]:
from langchain_core.prompts import PromptTemplate

In [6]:
# Chat completion model

from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are an {profession} expert on {topic}."),
    ("human", "Hello, Mr {profession}, can you please answer a question ?"),
    ("ai", "Sure !"),
    ("human", "{user_input}."),
])

messages = chat_template.format_messages(
    profession="historian",
    topic="The Kennedy family",
    user_input="How many grandchildren does the Kennedy family have?"
)

response = chatModel.invoke(messages)
print(response.content)

As of my last knowledge, the Kennedy family has a total of 35 grandchildren. This number may have changed since then, so I recommend checking the most recent information to confirm.


# Few Shot Promting

In [7]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate
examples = [
    {
        "input": "hi", 
        "output": "¡hola!"
    },
    {
        "input": "bye", 
        "output": "¡adiós!"
    }
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that translates English to Spanish."),
    ("human", "{input}"),
    few_shot_prompt
])

# Chains

In [13]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate
examples = [
    {
        "input": "hi", 
        "output": "¡hola!"
    },
    {
        "input": "bye", 
        "output": "¡adiós!"
    }
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that translates English to Spanish."),
    ("human", "{input}"),
    few_shot_prompt
])

chain = final_prompt | chatModel

response = chain.invoke({"input": "How are you?"})
print(response.content)


¡Hola! ¿En qué puedo ayudarte hoy?


In [ ]:
# Langchain Expression Language

# Output Parsers

In [19]:
from langchain_openai import OpenAI

llmModel = OpenAI(model="gpt-3.5-turbo")

In [15]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [21]:
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers.json import SimpleJsonOutputParser

json_prompt = PromptTemplate.from_template(
    "Return a JSON object with an 'answer' key that answers the following question: {input}"
)

json_parser = SimpleJsonOutputParser()

json_chain = json_prompt | chatModel | json_parser

In [29]:
response = json_chain.invoke({"input": "What is the capital of France?"})
response

{'answer': 'Paris'}

In [28]:
type(response)

dict

### Optionally, you can use Pydantic to define a custom output format

In [30]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field


In [31]:
# Define a Pydantic model for the JSON output
class Joke(BaseModel):
    setup: str = Field(description="The question to setup the joke")
    punchline: str = Field(description="The answer of the joke")

In [ ]:
# Define the parser reffering to the Joke Pydantic objet
parser = JsonOutputParser(pydantic_object=Joke)

# Add the parser format instructions in the prompt definition
prompt = PromptTemplate(
    template="Answer the user query.\n\n{format_instructions}",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

#Create a chain with the prompt and the parser
chain = prompt | chatModel | parser

chain.invoke({"query": "Tell me a joke."})

{'setup': 'Why did the scarecrow win an award?',
 'punchline': 'Because he was outstanding in his field!'}